## データフレーム変換関数

In [ ]:
import requests
import time
import pandas as pd

def gen_boj_dataframe(
    db:str,
    codes:list[str],
    startdate:str,
    enddate:str) -> pd.DataFrame:
    url = "https://www.stat-search.boj.or.jp/api/v1/getDataCode"
    params = {
        "DB": db,
        "CODE": ",".join(codes),
        "FORMAT": "JSON",
        "LANG":"JP",
        "STARTDATE": startdate,
        "ENDDATE": enddate
        }

    # ページネーションで最後のページを取得するまで、繰り返し処理を実行
    # Web APIからの取得データを格納
    all_results = []

    while True:
        # Web APIへのリクエストと例外処理。エラー時は空のデータフレームを返して終了
        try:
            response = requests.get(url, params=params, timeout=30)
            response.raise_for_status()
            response_data = response.json()
        except requests.exceptions.Timeout:
            print("Web APIから30秒以内に応答がありませんでした。")
            return pd.DataFrame()
        except requests.exceptions.ConnectionError:
            print("Web APIに接続できませんでした。")
            return pd.DataFrame()
        except requests.exceptions.HTTPError as error:
            print(f"HTTPエラーが発生しました: {error}")

            content_type = response.headers.get("Content-Type", "")
            if "application/json" in content_type:
                error_data = response.json()
                print(error_data["MESSAGE"])
            return pd.DataFrame()
        except requests.exceptions.JSONDecodeError:
            print("レスポンスをJSONとして読み込めませんでした。")
            return pd.DataFrame()
        else:
            if response_data["MESSAGEID"] == "M181030I":
                print(response_data["MESSAGE"])
                return pd.DataFrame()

            all_results.extend(response_data["RESULTSET"])
            next_position = response_data["NEXTPOSITION"]
            if next_position is None:
                break

            params["STARTPOSITION"] = next_position
            time.sleep(1)

    # 取得結果をデータフレームに変換
    df = pd.DataFrame(all_results)
    result_df = pd.DataFrame()

    for i in range(len(df)):
        values_df = pd.DataFrame(df.loc[i,"VALUES"])
        values_df = values_df.assign(
            SERIES_CODE=df.loc[i, "SERIES_CODE"],
            NAME_OF_TIME_SERIES_J=df.loc[i, "NAME_OF_TIME_SERIES_J"],
            CATEGORY_J=df.loc[i, "CATEGORY_J"]
        )
        result_df = pd.concat([result_df, values_df])

    result_df = result_df[["SERIES_CODE","NAME_OF_TIME_SERIES_J","CATEGORY_J","SURVEY_DATES","VALUES"]]

    result_df = result_df.reset_index(drop=True)

    result_df = result_df.astype({'SURVEY_DATES': str})

    return result_df

## データフレーム変換関数の実行とCSVファイルとして保存

In [ ]:
from google.colab import files

codes = [
    "PRCG20_2200010001",
    "PRCG20_2202010001",
    "PRCG20_2202110001",
    "PRCG20_2202210001",
    "PRCG20_2202310001",
]

pr01_df = gen_boj_dataframe("PR01", codes, "202504", "202603")

pr01_df.to_csv("pr01_df.csv", index=False)

files.download("pr01_df.csv")